# Medallion Architecture (Bronze / Silver / Gold)

**Topics in this module:**
1. What is Medallion Architecture and why does it exist?
2. **Bronze** Layer — raw data, as received
3. **Silver** Layer — clean, validated, and standardized data
4. **Gold** Layer — aggregated data, ready for business use
5. The complete flow: how the three layers are linked


---
## Important Note: Free Edition / Community Edition (Serverless)

We are using the **Serverless Free Edition**. Here's what applies in this environment:

### What you CAN do (what we'll do in this module)
- Create three Delta tables (`bronze_*`, `silver_*`, `gold_*`) inside a **Unity Catalog Schema**.
- Write data between layers using both pure SQL and PySpark DataFrames—fully supported in Serverless.
- Simulate raw data ingestion by **writing CSV/JSON files to a Volume** (following the same approach used in Module 20).

### What's different compared to the book or a production environment
- The book typically uses `dbfs:/mnt/...` paths and sometimes **one schema per layer**
  (`bronze`, `silver`, `gold` as separate databases). Since **DBFS root and mounts are deprecated**
  and unavailable in the Free Edition, we'll use **Volumes**
  (`/Volumes/catalog/schema/volume/...`) and a **single schema** with table prefixes
  (`bronze_`, `silver_`, `gold_`). The Medallion architecture remains exactly the same.
- For Bronze data lineage, `input_file_name()` **is not supported with Unity Catalog**
  (which is what the Free Edition uses). Instead, we'll use the hidden column
  **`_metadata.file_path`**, which is the supported approach and provides the same result.
- We won't build an orchestrated pipeline (Jobs or Delta Live Tables). Instead, we'll execute
  each layer **manually, cell by cell**, so you can clearly see the transformation at every step.
  In a production environment, these transformations would typically run as a Job or in Delta Live Tables.
- We'll work with a small social media posts dataset so each layer fits on the screen and the
  transformations are easy to understand. The same techniques scale to millions of records.

### The demonstration pattern

1. **Bronze:** Ingest raw, messy data (strings, inconsistent date formats, duplicates) → stored without modification.
2. **Silver:** Clean, cast data types, remove duplicates, and standardize → trusted, analytics-ready table.
3. **Gold:** Aggregate data into business metrics (KPIs and summaries) → table ready for dashboards and machine learning.

The core concept is identical to what you'll encounter on the certification exam (progressively higher data quality: raw → clean → business-ready); we're simply adapting the implementation to the Serverless environment.

---


---
## 0. Setup

- Create catalog/schema/volume if they don't already exist (same pattern as modules 18 and 20).
- Define a `landing/` folder within the Volume where the raw data is stored.

In [0]:
# Environment variables
catalog = "main"
schema  = "medallion"
volume  = "raw_data"

base_path   = f"/Volumes/{catalog}/{schema}/{volume}"
landing     = f"{base_path}/landing"          # where the raw files fall (the source)

print("Base path :", base_path)
print("Landing   :", landing)

Base path : /Volumes/main/medallion/raw_data
Landing   : /Volumes/main/medallion/raw_data/landing


In [0]:
# Create catalog/schema/volume structure if it does not exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {catalog}.{schema}.{volume}")

# Default schema for using %sql without a prefix
spark.sql(f"USE {catalog}.{schema}")
print(f"List structure. Using {catalog}.{schema}")

List structure. Using main.medallion


In [0]:
# Helper: write a text file to a Volume
# In Serverless, we can write directly to /Volumes/... using Python's open().
# We'll use this helper to "drop" files into the landing area and simulate
# the arrival of raw data.

import os

def drop_file(folder, filename, content):
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/{filename}"
    with open(path, "w") as f:
        f.write(content)
    print(f"  + file created: {path}")

print("Helper drop_file() is ready")

Helper drop_file() is ready


---
## 1. What is the Medallion Architecture?

The **Medallion Architecture** is a **design pattern** for organizing data in a **Lakehouse** into **three successive layers**, where **data quality and business value increase** at each stage.

| Layer | Also called | Contains | Answers the question |
|---|---|---|---|
| **Bronze** | raw | Raw data exactly as it arrives from the source | What data arrived? |
| **Silver** | clean / validated | Cleaned, standardized, typed, and deduplicated data | What data is trustworthy? |
| **Gold** | business / curated | Aggregations, KPIs, and business-ready tables | What does the business need? |

**Why use it?**

- **Data lineage:** Bronze preserves the original data unchanged, so you can always reprocess it.
- **Separation of responsibilities:** Data cleansing (Silver) and business aggregation (Gold) are independent stages.
- **Reusability:** Multiple Gold tables can be built from the same Silver layer.

Data flows in a single direction: **Bronze → Silver → Gold**. In this module, we'll build all three layers.
---


---
## 2. Bronze Layer — Raw Data

The Bronze layer is a **faithful copy** of the data as it arrives from the source. The guiding principle is: **do not transform the data**.

- Ingest the data exactly as received, even if it's messy (strings instead of numeric types, inconsistent date formats, duplicate records).
- Typically **add ingestion metadata** such as `_ingested_at` and `_source_file` to support data lineage and auditing.
- It serves as the **source of truth**: if something goes wrong downstream, you can always reprocess the data from Bronze.

First, we'll drop a raw CSV file into the landing area. Notice that it's **intentionally messy**:

- `likes` and `post_id` are stored as text.
- Dates use the `DD/MM/YYYY` format.
- Platform names have inconsistent capitalization.
- One record is **duplicated** (`P002` appears twice).

---


In [0]:
# Drop a raw CSV file into the landing area
# Intentionally messy data: numeric values stored as text, dates in DD/MM/YYYY,
# inconsistent platform names (Instagram / instagram / INSTAGRAM),
# and a DUPLICATE record (P002).

csv_header = "post_id|platform|created_at|impressions|likes"

drop_file(
    landing,
    "posts_001.csv",
    csv_header + "\n" +
    "P001|Instagram|17/06/2026|1000|50\n" +
    "P002|instagram|17/06/2026|500|25\n" +
    "P002|instagram|17/06/2026|500|25\n" +   # <- duplicate row
    "P003|INSTAGRAM|18/06/2026|2000|140\n" +
    "P004|TikTok|18/06/2026|800|90\n"
)

print("\nRaw file successfully dropped into the landing area.")


  + file created: /Volumes/main/medallion/raw_data/landing/posts_001.csv

Raw file successfully dropped into the landing area.


In [0]:
# BRONZE: ingest the data AS-IS, WITHOUT transformations
# Read the raw CSV file. The key principle of the Bronze layer is:
# inferSchema=False -> EVERYTHING is read as a string.
# We don't "fix" anything yet; we simply capture the data exactly as it arrived.

from pyspark.sql.functions import current_timestamp, col

raw_df = (
    spark.read
        .option("header", "true")
        .option("delimiter", "|")
        .option("inferSchema", "false")   # Bronze = no type casting: everything is string
        .csv(landing)
)

# Add ingestion metadata (data lineage) -> Bronze best practice
bronze_df = (
    raw_df
        .withColumn("_ingested_at", current_timestamp())
        # In Unity Catalog, input_file_name() is NOT supported.
        # The supported approach is the hidden _metadata.file_path column.
        .withColumn("_source_file", col("_metadata.file_path"))
)

# Persist as a Delta table (the Bronze layer).
# Bronze is APPEND-ONLY: each ingestion ADDS new records instead of replacing
# the existing data (it serves as the source of truth).
bronze_df.write.mode("append").saveAsTable("bronze_posts")

print("Data successfully appended to bronze_posts.")

Data successfully appended to bronze_posts.


In [0]:
%sql
-- Bronze: the data exactly as it arrived (duplicate and all)
-- Note: 5 rows (includes duplicate P002), all as a string, date DD/MM/YYYY.
SELECT post_id, platform, created_at, impressions, likes
FROM   bronze_posts
ORDER  BY post_id

post_id,platform,created_at,impressions,likes
P001,Instagram,17/06/2026,1000,50
P002,instagram,17/06/2026,500,25
P002,instagram,17/06/2026,500,25
P003,INSTAGRAM,18/06/2026,2000,140
P004,TikTok,18/06/2026,800,90


> **Exam Point:** in Bronze **it doesn't clean up or deduplicate**. Its job is to preserve the original data. That's why you see the duplicate of `P002` and the types as strings: that's fixed in Silver, not here. Bronze prioritizes **traceability** over cleanup.


---
## 3. Silver Layer (Clean and Reliable Data)

Silver is the **first layer where the data is processed**. This is where the important steps happen:

- **Type Casting:** `impressions` and `likes` from string → integer; `created_at` → date.
- **Standardization:** normalize `platform` (everything to a consistent format, e.g., `Instagram`).
- **Deduplication:** remove the duplicate row from `P002`.
- **Validation:** discard (or mark) any invalid rows.

The result is a **reliable and reusable** table: any subsequent analysis starts from here.

In [0]:
# SILVER: clean, cast, standardize, and deduplicate the data

from pyspark.sql.functions import col, to_date, initcap, trim

silver_df = (
    spark.table("bronze_posts")
        # 1) Cast data types: string -> int / date
        .withColumn("impressions", col("impressions").cast("int"))
        .withColumn("likes", col("likes").cast("int"))
        .withColumn("created_at", to_date(col("created_at"), "dd/MM/yyyy"))  # DD/MM/YYYY -> DATE

        # 2) Standardize platform names:
        #    'instagram' / 'INSTAGRAM' -> 'Instagram'
        .withColumn("platform", initcap(trim(col("platform"))))

        # 3) Keep only the business columns
        #    (drop Bronze ingestion metadata)
        .select("post_id", "platform", "created_at", "impressions", "likes")

        # 4) Remove duplicate records
        #    (the duplicated P002 row is collapsed into one)
        .dropDuplicates()
)

silver_df.write.mode("overwrite").saveAsTable("silver_posts")

print("Table silver_posts created.")

Table silver_posts created.


In [0]:
%sql
-- Silver: Clean data. P002 is no longer duplicated, correct types
-- 4 rows (duplicate disappeared), standardized platform, created_at is DATE.
SELECT post_id, platform, created_at, impressions, likes
FROM   silver_posts
ORDER  BY post_id

post_id,platform,created_at,impressions,likes
P001,Instagram,2026-06-17,1000,50
P002,Instagram,2026-06-17,500,25
P003,Instagram,2026-06-18,2000,140
P004,Tiktok,2026-06-18,800,90


In [0]:
# Check the type change (Bronze string -> Silver typed)
print("Bronze (all string):")
spark.table("bronze_posts").select("impressions", "likes", "created_at").printSchema()

print("Silver (real types):")
spark.table("silver_posts").select("impressions", "likes", "created_at").printSchema()

Bronze (all string):
root
 |-- impressions: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- created_at: string (nullable = true)

Silver (real types):
root
 |-- impressions: integer (nullable = true)
 |-- likes: integer (nullable = true)
 |-- created_at: date (nullable = true)



> **Exam tip:** The Silver layer is where data becomes **trustworthy**: data types are corrected, duplicates are removed, and values are standardized. It is the layer most commonly reused across teams because the data is clean while still remaining **granular** (one row per post), without business aggregations—that is the responsibility of the Gold layer.

> **Implementation detail:** `initcap` capitalizes the first letter of each word, so `"TikTok"` becomes `"Tiktok"`. This is Spark's actual behavior. In a production environment, you would typically use an explicit mapping (for example, a lookup table or a `CASE WHEN` statement) to preserve the exact spelling of each platform name.



## 4. Gold Layer (Business-Ready Data)

The Gold layer **no longer operates at the individual row level**. Instead, it contains **aggregated business metrics** that are directly consumed by dashboards, reports, and machine learning models.

In the Silver layer, we had **one row per post**. In the Gold layer, we answer business questions such as:

> *"How many impressions and what engagement did each platform generate per day?"*

To answer this, we aggregate the data using `GROUP BY platform, created_at` and calculate a business **KPI**: the `engagement_rate` (`likes / impressions`).

> **In Gold, individual publications are no longer the focus; instead, the metrics consumed by the business are what matter.**


In [0]:
%sql
-- GOLD: Business aggregation (KPIs per platform and day)
-- From multiple rows (individual posts) in Silver -> few rows with metrics.
CREATE OR REPLACE TABLE gold_daily_engagement AS
SELECT
  platform,
  created_at                                    AS date,
  COUNT(*)                                      AS total_posts,
  SUM(impressions)                              AS total_impressions,
  SUM(likes)                                    AS total_likes,
  ROUND(SUM(likes) / SUM(impressions) * 100, 2) AS engagement_rate_pct
FROM   silver_posts
GROUP  BY platform, created_at
ORDER  BY platform, date

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Gold: the final table ready for a dashboard
-- A few rows, one per (platform, day), with the KPI already calculated.
SELECT * FROM gold_daily_engagement ORDER BY platform, date

platform,date,total_posts,total_impressions,total_likes,engagement_rate_pct
Instagram,2026-06-17,2,1500,75,5.0
Instagram,2026-06-18,1,2000,140,7.0
Tiktok,2026-06-18,1,800,90,11.25


> **Exam tip:** In the Gold layer, **individual records are no longer the focus**. Instead, the emphasis is on the **business metrics** consumed by dashboards, reports, and machine learning models. It is common to have **multiple Gold tables** (one per use case, such as a dashboard, ML model, or business report), all built from the **same Silver layer**. Gold prioritizes **business value** over data granularity.


---
## 5. The Complete Data Flow at a Glance

We've now walked through all three layers. Here's how the data evolves for record `P002` (the one that originally contained duplicate and inconsistent data):

| Layer | post_id | platform | created_at | likes | Rows | Status |
|---|---|---|---|---|---|---|
| **Bronze** | `"P002"` | `"instagram"` | `"17/06/2026"` | `"25"` (string) | Duplicate ×2 | Raw |
| **Silver** | `P002` | `Instagram` | `2026-06-17` (date) | `25` (int) | 1 row | Clean |
| **Gold** | — | `Instagram` | `2026-06-17` | (aggregated) | Daily aggregate | Business-ready |

The following query counts the number of rows in each layer to illustrate the transformation:

- **Bronze** retains the duplicate record.
- **Silver** removes the duplicate.
- **Gold** aggregates the data into business metrics.


In [0]:
# Count the number of rows in each layer to visualize the effect of each transformation

b = spark.table("bronze_posts").count()
s = spark.table("silver_posts").count()
g = spark.table("gold_daily_engagement").count()

print(f"Bronze (raw, includes duplicate) : {b} rows")
print(f"Silver (clean, deduplicated)     : {s} rows")
print(f"Gold (business aggregates)       : {g} rows")
print()

print("Bronze -> Silver: the duplicate P002 record was removed.")
print("Silver -> Gold : individual posts were aggregated by (platform, date).")


Bronze (raw, includes duplicate) : 5 rows
Silver (clean, deduplicated)     : 4 rows
Gold (business aggregates)       : 3 rows

Bronze -> Silver: the duplicate P002 record was removed.
Silver -> Gold : individual posts were aggregated by (platform, date).


---
## Clean Up

In [0]:
def clean_up():
    print("Dropping tables...")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.bronze_posts")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.silver_posts")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.gold_daily_engagement")

    print("Removing files from the Volume...")
    dbutils.fs.rm(landing, True)

    print("Dropping schema...")
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE")

    print("Done \u2713")

In [0]:
# clean_up()